# Multi-Voice Activity Detection — Single File Analysis

This notebook runs both the **Traditional (signal-processing)** and **DNN** MVAD models on a single WAV file and displays prediction waveforms for comparison.

In [ ]:
import sys
import numpy as np
import soundfile as sf
import scipy.io

# Import everything from mvad_test.py
# (mvad_test.py forces matplotlib backend to 'Agg' at import time,
#  so we must re-set it to 'inline' AFTER the import)
from mvad_test import (
    MultivoiceVAD,
    load_dnn_model,
    dnn_predict_file,
)

import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
%matplotlib inline
import matplotlib.pyplot as plt

print('Imports OK, matplotlib backend:', matplotlib.get_backend())

## 1. Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
# INPUT_WAV   = 'dumps/example_3_SHU1300__TMR_cfg_single_talker_mobile_with_bbldagc_48kHz_txfeOut.wav'
INPUT_WAV   = 'inputs/example_18.wav'
GT_MAT      = 'inputs/example_18_manualVAD.mat'   # ground truth (sample-level)

# ── DNN model selection ───────────────────────────────────────────────────
# DNN_TYPE: 'mel'  → original mel-feature DNN (mvad_dnn_model_ep47.pt)
#           'i'    → VAD_I_C2_L8 raw-waveform 1D conv (mvad_dnn_i_model_ep25.pt)
#           'h'    → VAD_H_C24_L10 raw-waveform 1D conv (mvad_dnn_h_model_ep31.pt)
#           'v2'   → MVAD_V2 DS-Conv1D mel-feature model (mvad_dnn_v2_model_ep46.pt)
DNN_TYPE    = 'v2'
DNN_MODEL_MEL = 'mvad_dnn_model_ep47.pt'
DNN_MODEL_I   = 'mvad_dnn_i_model_ep25.pt'
DNN_MODEL_H   = 'mvad_dnn_h_model_ep31.pt'
DNN_MODEL_V2  = 'mvad_dnn_v2_model_ep46.pt'
DNN_MODEL     = {'mel': DNN_MODEL_MEL, 'i': DNN_MODEL_I, 'h': DNN_MODEL_H, 'v2': DNN_MODEL_V2}[DNN_TYPE]

# ── Traditional VAD parameters (defaults from mvad_test.py) ───────────────
FRAME_MS              = 30
HOP_MS                = 10
ENERGY_THRESHOLD_DB   = -40
PITCH_CONF_THRESH     = 0.25
YIN_THRESHOLD         = 0.15
SECONDARY_PITCH_CONF  = 0.20
SF_THRESHOLD          = 0.30
OVERLAP_THRESHOLD     = 0.38
CONTEXT_FRAMES        = 3
MEDIAN_FILTER         = 7
F0_MIN                = 80
F0_MAX                = 400

## 2. Load audio

In [ ]:
signal, sr = sf.read(INPUT_WAV, dtype='float64')
if signal.ndim > 1:
    signal = np.mean(signal, axis=1)

duration = len(signal) / sr
print(f'File     : {INPUT_WAV}')
print(f'SR       : {sr} Hz')
print(f'Duration : {duration:.2f} s')
print(f'Samples  : {len(signal):,}')

# ── Load ground truth ────────────────────────────────────────────────────
gt_data = scipy.io.loadmat(GT_MAT)
gt_vad  = gt_data['vad'].flatten()   # sample-level: 0=silence, 1=single, 2=overlap
print(f'GT MAT   : {GT_MAT}  ({len(gt_vad):,} samples)')
for lab, name in [(0, 'Silence'), (1, 'Single'), (2, 'Overlap')]:
    cnt = np.sum(gt_vad == lab)
    print(f'  {name:20s}: {cnt/sr:.2f} s  ({cnt/len(gt_vad)*100:.1f}%)')

## 3. Run Traditional VAD

In [ ]:
vad = MultivoiceVAD(
    sr=sr,
    frame_ms=FRAME_MS,
    hop_ms=HOP_MS,
    energy_threshold_db=ENERGY_THRESHOLD_DB,
    pitch_conf_thresh=PITCH_CONF_THRESH,
    yin_threshold=YIN_THRESHOLD,
    secondary_pitch_conf=SECONDARY_PITCH_CONF,
    spectral_flatness_overlap=SF_THRESHOLD,
    overlap_threshold=OVERLAP_THRESHOLD,
    context_frames=CONTEXT_FRAMES,
    median_filter_size=MEDIAN_FILTER,
    f0_min=F0_MIN,
    f0_max=F0_MAX,
)

trad_labels, trad_features = vad.process(signal)

n_trad = len(trad_labels)
hop_s = vad.hop_len / sr
trad_times = np.arange(n_trad) * hop_s

print(f'Traditional VAD: {n_trad} frames')
for lab, name in [(0, 'Silence'), (1, 'Single'), (2, 'Overlap')]:
    cnt = np.sum(trad_labels == lab)
    print(f'  {name:20s}: {cnt:6d} frames  ({cnt/n_trad*100:5.1f}%)  {cnt*hop_s:.2f}s')

## 4. Run DNN Model

In [ ]:
# ── VAD_I_C2_L8 model definition & helpers (for DNN_TYPE == 'i') ──────────
import torch
import torch.nn as nn
from math import gcd
from scipy.signal import resample_poly

TARGET_SAMPLE_RATE_I = 16_000
DECIMATION_FACTOR_I  = 128       # total network decimation (2^7)
NUM_CLASSES_I        = 3


class ResCbr1dGen(nn.Module):
    """Residual Conv-BatchNorm-ReLU 1D block (valid convolution, no padding)."""
    def __init__(self, in_channels, out_channels, kernel_size, stride=1,
                 residual=False):
        super().__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size,
                              stride=stride, padding=0, bias=False)
        self.bn = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.use_residual = residual and (in_channels == out_channels)
        self.stride = stride

    def forward(self, x):
        out = self.relu(self.bn(self.conv(x)))
        if self.use_residual:
            skip = x
            if self.stride > 1:
                skip = skip[:, :, ::self.stride]
            out_len = out.size(2)
            skip = skip[:, :, -out_len:]
            out = out + skip
        return out


class VAD_I_C2_L8(nn.Module):
    """1D Conv Encoder-Decoder for VAD (from vad_i_c2_l8_architecture.html)."""
    def __init__(self, num_classes=NUM_CLASSES_I):
        super().__init__()
        self.fwd = nn.ModuleList([
            ResCbr1dGen(1, 2, kernel_size=7, stride=2, residual=False),
            ResCbr1dGen(2, 4, kernel_size=7, stride=2, residual=False),
            ResCbr1dGen(4, 8, kernel_size=7, stride=2, residual=False),
        ])
        self.core = nn.ModuleList([
            ResCbr1dGen(8, 8, kernel_size=7, stride=2, residual=True),
            ResCbr1dGen(8, 8, kernel_size=9, stride=2, residual=True),
            ResCbr1dGen(8, 8, kernel_size=9, stride=2, residual=True),
            ResCbr1dGen(8, 8, kernel_size=9, stride=2, residual=True),
            ResCbr1dGen(8, 8, kernel_size=9, stride=1, residual=True),
            ResCbr1dGen(8, 8, kernel_size=9, stride=1, residual=True),
            ResCbr1dGen(8, 8, kernel_size=9, stride=1, residual=True),
        ])
        self.inv = nn.ModuleList([
            ResCbr1dGen(8, 4, kernel_size=9, stride=1, residual=False),
            ResCbr1dGen(4, num_classes, kernel_size=128, stride=1, residual=False),
        ])

    def forward(self, x):
        for layer in self.fwd:
            x = layer(x)
        for layer in self.core:
            x = layer(x)
        for layer in self.inv:
            x = layer(x)
        return x


def _resample_to_16k(audio, orig_sr):
    """Resample audio to 16 kHz."""
    if orig_sr == TARGET_SAMPLE_RATE_I:
        return audio.astype(np.float32)
    g = gcd(int(TARGET_SAMPLE_RATE_I), int(orig_sr))
    up = int(TARGET_SAMPLE_RATE_I) // g
    down = int(orig_sr) // g
    return resample_poly(audio, up, down).astype(np.float32)


def load_dnn_i_model(model_path, device=None):
    """Load a trained VAD_I_C2_L8 model from checkpoint."""
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ckpt = torch.load(str(model_path), map_location=device, weights_only=False)
    cfg = ckpt['config']
    model = VAD_I_C2_L8(num_classes=cfg.get('num_classes', NUM_CLASSES_I))
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device)
    model.eval()
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  DNN-I model loaded: arch={cfg["arch"]}, '
          f'decimation={cfg["decimation_factor"]}×, '
          f'params={n_params:,}, device={device}')
    if 'epoch' in ckpt:
        print(f'  Best epoch: {ckpt["epoch"]}')
    return model, cfg, device


def dnn_i_predict_file(audio, orig_sr, model, cfg, device):
    """
    Run VAD_I_C2_L8 inference on a full audio file.

    The audio is resampled to 16 kHz, then processed in overlapping chunks.
    Returns per-frame predictions at 125 Hz (8 ms hop).
    """
    audio_16k = _resample_to_16k(audio, orig_sr)
    dec = cfg.get('decimation_factor', DECIMATION_FACTOR_I)
    chunk_samples = cfg.get('chunk_samples', 48_000)

    # Compute output length for this chunk size (analytically)
    LAYER_DEFS = [
        (7, 2), (7, 2), (7, 2),
        (7, 2), (9, 2), (9, 2), (9, 2),
        (9, 1), (9, 1), (9, 1),
        (9, 1), (128, 1),
    ]
    out_len = chunk_samples
    for k, s in LAYER_DEFS:
        out_len = (out_len - k) // s + 1

    stride_samples = out_len * dec
    total_frames = len(audio_16k) // dec

    predictions = np.full(total_frames, -1, dtype=np.int32)

    with torch.no_grad():
        for start in range(0, len(audio_16k) - chunk_samples + 1,
                           stride_samples):
            chunk = audio_16k[start: start + chunk_samples]
            x = torch.from_numpy(chunk).float().unsqueeze(0).unsqueeze(0)
            x = x.to(device)                              # (1, 1, chunk_samples)
            logits = model(x)                              # (1, C, out_len)
            preds = logits.argmax(dim=1).squeeze(0).cpu().numpy()  # (out_len,)

            # Right-aligned: output frames correspond to the rightmost part
            end_sample = start + chunk_samples
            end_frame = end_sample // dec
            frame_start = end_frame - out_len
            lo = max(0, frame_start)
            hi = min(total_frames, end_frame)
            src_lo = lo - frame_start
            src_hi = src_lo + (hi - lo)
            predictions[lo:hi] = preds[src_lo:src_hi]

    # Fill any remaining frames (if audio shorter than one chunk)
    if np.any(predictions < 0):
        # Process the tail with the last possible chunk
        if len(audio_16k) >= chunk_samples:
            start = len(audio_16k) - chunk_samples
            chunk = audio_16k[start: start + chunk_samples]
            x = torch.from_numpy(chunk).float().unsqueeze(0).unsqueeze(0).to(device)
            logits = model(x)
            preds = logits.argmax(dim=1).squeeze(0).cpu().numpy()
            end_frame = len(audio_16k) // dec
            frame_start = end_frame - out_len
            lo = max(0, frame_start)
            hi = min(total_frames, end_frame)
            src_lo = lo - frame_start
            src_hi = src_lo + (hi - lo)
            mask = predictions[lo:hi] < 0
            predictions[lo:hi] = np.where(mask, preds[src_lo:src_hi],
                                          predictions[lo:hi])
        # Any still unfilled → silence
        predictions[predictions < 0] = 0

    return predictions


# ── VAD_H_C24_L10 model definition & helpers (for DNN_TYPE == 'h') ─────────

TARGET_SAMPLE_RATE_H = 16_000
DECIMATION_FACTOR_H  = 64        # total network decimation (2^6)
NUM_CLASSES_H        = 3

LAYER_DEFS_H = [
    (128, 2), (9, 2), (9, 2), (9, 2), (9, 2),           # fwd L1-L5
    (9, 2),                                                # core L6 (stride-2)
    (9, 1), (9, 1), (9, 1), (9, 1), (9, 1), (9, 1),     # core L7-L12 (stride-1)
    (9, 1), (9, 1), (9, 1),                               # inv L13-L15
    (128, 1),                                              # inv L16
]


class VAD_H_C24_L10(nn.Module):
    """1D Conv Encoder-Decoder for VAD (from vad_h_c24_l10_architecture.html).

    16 layers, 64× decimation (250 Hz output), mixed alignment (center L1-L3, right L4-L16).
    Channels: 1→2→4→8→16→32 (encoder) → 32 (core) → 16→8→4→C (decoder).
    """
    def __init__(self, num_classes=NUM_CLASSES_H):
        super().__init__()
        self.fwd = nn.ModuleList([
            ResCbr1dGen(1,  2,  kernel_size=128, stride=2, residual=False),  # L1
            ResCbr1dGen(2,  4,  kernel_size=9,   stride=2, residual=False),  # L2
            ResCbr1dGen(4,  8,  kernel_size=9,   stride=2, residual=False),  # L3
            ResCbr1dGen(8,  16, kernel_size=9,   stride=2, residual=False),  # L4
            ResCbr1dGen(16, 32, kernel_size=9,   stride=2, residual=False),  # L5
        ])
        self.core = nn.ModuleList([
            ResCbr1dGen(32, 32, kernel_size=9, stride=2, residual=True),   # L6
            ResCbr1dGen(32, 32, kernel_size=9, stride=1, residual=True),   # L7
            ResCbr1dGen(32, 32, kernel_size=9, stride=1, residual=True),   # L8
            ResCbr1dGen(32, 32, kernel_size=9, stride=1, residual=True),   # L9
            ResCbr1dGen(32, 32, kernel_size=9, stride=1, residual=True),   # L10
            ResCbr1dGen(32, 32, kernel_size=9, stride=1, residual=True),   # L11
            ResCbr1dGen(32, 32, kernel_size=9, stride=1, residual=True),   # L12
        ])
        self.inv = nn.ModuleList([
            ResCbr1dGen(32, 16,         kernel_size=9,   stride=1, residual=False),  # L13
            ResCbr1dGen(16, 8,          kernel_size=9,   stride=1, residual=False),  # L14
            ResCbr1dGen(8,  4,          kernel_size=9,   stride=1, residual=False),  # L15
            ResCbr1dGen(4,  num_classes, kernel_size=128, stride=1, residual=False), # L16
        ])

    def forward(self, x):
        for layer in self.fwd:
            x = layer(x)
        for layer in self.core:
            x = layer(x)
        for layer in self.inv:
            x = layer(x)
        return x


def load_dnn_h_model(model_path, device=None):
    """Load a trained VAD_H_C24_L10 model from checkpoint."""
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ckpt = torch.load(str(model_path), map_location=device, weights_only=False)
    cfg = ckpt['config']
    model = VAD_H_C24_L10(num_classes=cfg.get('num_classes', NUM_CLASSES_H))
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device)
    model.eval()
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  DNN-H model loaded: arch={cfg["arch"]}, '
          f'decimation={cfg["decimation_factor"]}×, '
          f'params={n_params:,}, device={device}')
    if 'epoch' in ckpt:
        print(f'  Best epoch: {ckpt["epoch"]}')
    return model, cfg, device


def dnn_h_predict_file(audio, orig_sr, model, cfg, device):
    """
    Run VAD_H_C24_L10 inference on a full audio file.

    The audio is resampled to 16 kHz, then processed in overlapping chunks.
    Returns per-frame predictions at 250 Hz (4 ms hop).
    """
    audio_16k = _resample_to_16k(audio, orig_sr)
    dec = cfg.get('decimation_factor', DECIMATION_FACTOR_H)
    chunk_samples = cfg.get('chunk_samples', 48_000)

    # Compute output length analytically
    out_len = chunk_samples
    for k, s in LAYER_DEFS_H:
        out_len = (out_len - k) // s + 1

    stride_samples = out_len * dec
    total_frames = len(audio_16k) // dec

    predictions = np.full(total_frames, -1, dtype=np.int32)

    with torch.no_grad():
        for start in range(0, len(audio_16k) - chunk_samples + 1,
                           stride_samples):
            chunk = audio_16k[start: start + chunk_samples]
            x = torch.from_numpy(chunk).float().unsqueeze(0).unsqueeze(0)
            x = x.to(device)                              # (1, 1, chunk_samples)
            logits = model(x)                              # (1, C, out_len)
            preds = logits.argmax(dim=1).squeeze(0).cpu().numpy()  # (out_len,)

            # Right-aligned: output frames correspond to the rightmost part
            end_sample = start + chunk_samples
            end_frame = end_sample // dec
            frame_start = end_frame - out_len
            lo = max(0, frame_start)
            hi = min(total_frames, end_frame)
            src_lo = lo - frame_start
            src_hi = src_lo + (hi - lo)
            predictions[lo:hi] = preds[src_lo:src_hi]

    # Fill any remaining frames (if audio shorter than one chunk)
    if np.any(predictions < 0):
        if len(audio_16k) >= chunk_samples:
            start = len(audio_16k) - chunk_samples
            chunk = audio_16k[start: start + chunk_samples]
            x = torch.from_numpy(chunk).float().unsqueeze(0).unsqueeze(0).to(device)
            logits = model(x)
            preds = logits.argmax(dim=1).squeeze(0).cpu().numpy()
            end_frame = len(audio_16k) // dec
            frame_start = end_frame - out_len
            lo = max(0, frame_start)
            hi = min(total_frames, end_frame)
            src_lo = lo - frame_start
            src_hi = src_lo + (hi - lo)
            mask = predictions[lo:hi] < 0
            predictions[lo:hi] = np.where(mask, preds[src_lo:src_hi],
                                          predictions[lo:hi])
        predictions[predictions < 0] = 0

    return predictions


# ── MVAD_V2 (DS-Conv1D) model definition & helpers (for DNN_TYPE == 'v2') ──

from scipy.signal import stft as scipy_stft

# Mel-filterbank constants (must match train_mvad_dnn_v2.py)
V2_SAMPLE_RATE = 48_000
V2_FRAME_MS = 10
V2_HOP_SAMPLES = int(V2_SAMPLE_RATE * V2_FRAME_MS / 1000)          # 480
V2_ANALYSIS_WINDOW_MS = 25
V2_ANALYSIS_WINDOW_SAMPLES = int(V2_SAMPLE_RATE * V2_ANALYSIS_WINDOW_MS / 1000)  # 1200
V2_N_FFT = 2048
V2_N_MELS = 40
V2_FMIN = 80.0
V2_FMAX = 8000.0
V2_NUM_CLASSES = 3


def _hz_to_mel(hz):
    return 2595.0 * np.log10(1.0 + np.asarray(hz, dtype=np.float64) / 700.0)


def _mel_to_hz(mel):
    return 700.0 * (10.0 ** (np.asarray(mel, dtype=np.float64) / 2595.0) - 1.0)


def _create_mel_filterbank(sr, n_fft, n_mels, fmin=0.0, fmax=None):
    if fmax is None:
        fmax = sr / 2.0
    n_freqs = n_fft // 2 + 1
    mel_points = np.linspace(_hz_to_mel(fmin), _hz_to_mel(fmax), n_mels + 2)
    hz_points = _mel_to_hz(mel_points)
    bins = np.round(hz_points * n_fft / sr).astype(int)
    bins = np.clip(bins, 0, n_freqs - 1)
    fb = np.zeros((n_mels, n_freqs), dtype=np.float64)
    for m in range(n_mels):
        left, centre, right = bins[m], bins[m + 1], bins[m + 2]
        if centre > left:
            fb[m, left:centre + 1] = np.linspace(0.0, 1.0, centre - left + 1)
        if right > centre:
            fb[m, centre:right + 1] = np.linspace(1.0, 0.0, right - centre + 1)
    return fb.astype(np.float32)


_v2_mel_fb_cache = {}

def _get_v2_mel_fb(sr=V2_SAMPLE_RATE):
    if sr not in _v2_mel_fb_cache:
        _v2_mel_fb_cache[sr] = _create_mel_filterbank(sr, V2_N_FFT, V2_N_MELS, V2_FMIN, V2_FMAX)
    return _v2_mel_fb_cache[sr]


def _compute_log_mel_v2(audio, sr=V2_SAMPLE_RATE):
    """Compute log mel-filterbank energies → (n_frames, N_MELS) float32."""
    mel_fb = _get_v2_mel_fb(sr)
    noverlap = V2_ANALYSIS_WINDOW_SAMPLES - V2_HOP_SAMPLES
    _, _, Zxx = scipy_stft(audio, fs=sr, window='hann',
                           nperseg=V2_ANALYSIS_WINDOW_SAMPLES,
                           noverlap=noverlap, nfft=V2_N_FFT)
    power = np.abs(Zxx) ** 2
    mel_energy = mel_fb @ power
    log_mel = np.log(np.maximum(mel_energy, 1e-10))
    return log_mel.T.astype(np.float32)


class DSConvBlock(nn.Module):
    """Depthwise-Separable Conv1d block with optional residual."""
    def __init__(self, ch_in, ch_out, kernel_size=15, residual=False, dropout=0.1):
        super().__init__()
        self.use_residual = residual and (ch_in == ch_out)
        padding = kernel_size // 2
        self.dw_conv = nn.Conv1d(ch_in, ch_in, kernel_size, padding=padding,
                                 groups=ch_in, bias=False)
        self.dw_bn = nn.BatchNorm1d(ch_in)
        self.pw_conv = nn.Conv1d(ch_in, ch_out, 1, bias=False)
        self.pw_bn = nn.BatchNorm1d(ch_out)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        identity = x
        x = self.relu(self.dw_bn(self.dw_conv(x)))
        x = self.dropout(self.relu(self.pw_bn(self.pw_conv(x))))
        if self.use_residual:
            x = x + identity
        return x


class MVAD_V2(nn.Module):
    """Lightweight DS-Conv1D model for Multi-Voice VAD (mel-feature input)."""
    def __init__(self, n_mels=V2_N_MELS, n_classes=V2_NUM_CLASSES, hidden_ch=64,
                 kernel_size=15, n_blocks=5, dropout=0.1):
        super().__init__()
        self.n_mels = n_mels
        self.n_classes = n_classes
        self.hidden_ch = hidden_ch
        self.kernel_size = kernel_size
        self.n_blocks = n_blocks
        self.input_proj = nn.Sequential(
            nn.Conv1d(n_mels, hidden_ch, 1, bias=False),
            nn.BatchNorm1d(hidden_ch),
            nn.ReLU(inplace=True),
        )
        self.blocks = nn.ModuleList()
        for i in range(n_blocks):
            is_last = (i == n_blocks - 1)
            ch_out = hidden_ch // 2 if is_last else hidden_ch
            use_res = (i > 0) and (not is_last)
            self.blocks.append(DSConvBlock(
                hidden_ch, ch_out, kernel_size,
                residual=use_res, dropout=dropout
            ))
        self.output_head = nn.Conv1d(hidden_ch // 2, n_classes, 1)

    def forward(self, x):
        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x)
        return self.output_head(x)


def load_dnn_v2_model(model_path, device=None):
    """Load a trained MVAD_V2 model from checkpoint."""
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ckpt = torch.load(str(model_path), map_location=device, weights_only=False)
    cfg = ckpt['config']
    model = MVAD_V2(
        n_mels=cfg.get('n_mels', V2_N_MELS),
        n_classes=cfg.get('num_classes', V2_NUM_CLASSES),
        hidden_ch=cfg.get('hidden_ch', 64),
        kernel_size=cfg.get('kernel_size', 15),
        n_blocks=cfg.get('n_blocks', 5),
        dropout=cfg.get('dropout', 0.1),
    )
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device)
    model.eval()
    n_params = sum(p.numel() for p in model.parameters())
    # Extract standardisation (mean/std fitted on training set)
    std_info = ckpt.get('standardisation', {})
    feat_mean = std_info.get('mean', None)
    feat_std = std_info.get('std', None)
    print(f'  DNN-V2 model loaded: arch={cfg["arch"]}, '
          f'hidden={cfg["hidden_ch"]}, k={cfg["kernel_size"]}, '
          f'blocks={cfg["n_blocks"]}, params={n_params:,}, device={device}')
    if 'epoch' in ckpt:
        print(f'  Best epoch: {ckpt["epoch"]}')
    return model, cfg, feat_mean, feat_std, device


def dnn_v2_predict_file(audio, orig_sr, model, cfg, feat_mean, feat_std, device):
    """
    Run MVAD_V2 inference on a full audio file.

    Computes mel features at the file's native sample rate,
    applies z-score normalisation, and runs the model.
    Returns per-frame predictions at 100 Hz (10 ms hop).
    """
    # Compute log mel features
    mel = _compute_log_mel_v2(audio, orig_sr)   # (n_frames, n_mels)

    # Z-score normalisation
    if feat_mean is not None and feat_std is not None:
        mel = (mel - feat_mean) / feat_std

    # (n_frames, n_mels) → (1, n_mels, n_frames) for Conv1d
    mel_t = torch.from_numpy(mel.T.astype(np.float32)).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(mel_t)                        # (1, n_classes, n_frames)
        predictions = logits.argmax(dim=1).squeeze(0).cpu().numpy()

    return predictions


print('VAD_I_C2_L8 + VAD_H_C24_L10 + MVAD_V2 model helpers defined.')

In [ ]:
if DNN_TYPE == 'h':
    # ── VAD_H_C24_L10 raw-waveform model ────────────────────────────────
    model, cfg, device = load_dnn_h_model(DNN_MODEL)
    dnn_labels = dnn_h_predict_file(signal, sr, model, cfg, device)

    dec = cfg.get('decimation_factor', DECIMATION_FACTOR_H)
    target_sr = cfg.get('target_sample_rate', TARGET_SAMPLE_RATE_H)
    dnn_hop_samples_16k = dec                   # in 16 kHz samples
    dnn_hop_s = dnn_hop_samples_16k / target_sr # seconds per frame (4 ms)
elif DNN_TYPE == 'i':
    # ── VAD_I_C2_L8 raw-waveform model ──────────────────────────────────
    model, cfg, device = load_dnn_i_model(DNN_MODEL)
    dnn_labels = dnn_i_predict_file(signal, sr, model, cfg, device)

    dec = cfg.get('decimation_factor', DECIMATION_FACTOR_I)
    target_sr = cfg.get('target_sample_rate', TARGET_SAMPLE_RATE_I)
    dnn_hop_samples_16k = dec                   # in 16 kHz samples
    dnn_hop_s = dnn_hop_samples_16k / target_sr # seconds per frame (8 ms)
elif DNN_TYPE == 'v2':
    # ── MVAD_V2 DS-Conv1D mel-feature model ───────────────────────────────
    model, cfg, v2_feat_mean, v2_feat_std, device = load_dnn_v2_model(DNN_MODEL)
    dnn_labels = dnn_v2_predict_file(signal, sr, model, cfg, v2_feat_mean, v2_feat_std, device)

    # V2 outputs at 100 Hz (10 ms hop) — same rate as the original mel DNN
    dnn_hop_s = V2_FRAME_MS / 1000.0   # 0.01 s
else:
    # ── Original mel-feature DNN ─────────────────────────────────────────
    model, cfg, feat_mean, feat_std, device = load_dnn_model(DNN_MODEL)
    dnn_labels = dnn_predict_file(
        signal, sr, model, cfg, feat_mean, feat_std, device
    )
    dnn_hop_samples = cfg.get('hop_samples', int(sr * 0.01))
    dnn_hop_s = dnn_hop_samples / sr

n_dnn = len(dnn_labels)
dnn_times = np.arange(n_dnn) * dnn_hop_s

print(f'DNN model: {n_dnn} frames (arch={cfg["arch"]}, type={DNN_TYPE})')
for lab, name in [(0, 'Silence'), (1, 'Single'), (2, 'Overlap')]:
    cnt = np.sum(dnn_labels == lab)
    print(f'  {name:20s}: {cnt:6d} frames  ({cnt/n_dnn*100:5.1f}%)  {cnt*dnn_hop_s:.2f}s')

## 5. Prediction Plots — Waveform + Coloured Timeline

Two panels:
1. **Audio waveform** (background: manual Ground Truth)
2. **DNN model** — coloured timeline (cyan=single, orange=overlap)

In [ ]:
from matplotlib.patches import Patch
from matplotlib.collections import BrokenBarHCollection

CLASS_COLORS = {0: None, 1: 'cyan', 2: 'orange'}
CLASS_ALPHA  = {0: 0.0, 1: 1.0, 2: 1.0}
CLASS_NAMES  = {0: 'Silence', 1: 'Single speaker', 2: 'Overlap'}


def _label_segments(labels, hop_sec):
    """Convert frame labels to list of (start_sec, duration_sec, label)."""
    segments = []
    if len(labels) == 0:
        return segments
    cur_label = labels[0]
    seg_start = 0.0
    for i in range(1, len(labels)):
        if labels[i] != cur_label:
            segments.append((seg_start, i * hop_sec - seg_start, cur_label))
            cur_label = labels[i]
            seg_start = i * hop_sec
    segments.append((seg_start, len(labels) * hop_sec - seg_start, cur_label))
    return segments


def _draw_timeline(ax, segments, y_bottom=0, height=1):
    """Draw coloured horizontal bars for each segment on *ax*."""
    for start, dur, lab in segments:
        if CLASS_COLORS[lab] is not None:
            ax.barh(y_bottom + height / 2, dur, height=height, left=start,
                    color=CLASS_COLORS[lab], alpha=CLASS_ALPHA[lab],
                    edgecolor='none', linewidth=0)


# ── Build segments ────────────────────────────────────────────────────────
trad_segs = _label_segments(trad_labels, hop_s)
dnn_segs  = _label_segments(dnn_labels,  dnn_hop_s)

# ── Figure ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(22, 13), sharex=True,
                         gridspec_kw={'height_ratios': [1, 1],
                                      'hspace': 0.15})
fig.suptitle(f'Multivoice VAD Predictions — {INPUT_WAV}', fontsize=28, fontweight='bold', y=0.985)

# ── Panel 1: Waveform with GT fill areas ─────────────────────────────────
t_sig = np.arange(len(signal)) / sr

# Ground truth fill colours (very light, behind the waveform)
GT_FILL = {0: (None,    0.0),      # silence  — no fill
           1: ('cyan',  1.0),     # single   — cyan
           2: ('orange', 1.0)}    # overlap  — orange

# Build GT segments from sample-level labels (run-length encoding)
def _gt_sample_segments(gt, sample_rate):
    """Convert sample-level GT to (start_sec, end_sec, label) segments."""
    segs = []
    if len(gt) == 0:
        return segs
    cur = gt[0]; s0 = 0
    for i in range(1, len(gt)):
        if gt[i] != cur:
            segs.append((s0 / sample_rate, i / sample_rate, int(cur)))
            cur = gt[i]; s0 = i
    segs.append((s0 / sample_rate, len(gt) / sample_rate, int(cur)))
    return segs

gt_segs = _gt_sample_segments(gt_vad, sr)
ymin = -1.05 * np.max(np.abs(signal))
ymax =  1.05 * np.max(np.abs(signal))
for t0, t1, lab in gt_segs:
    fc, fa = GT_FILL[lab]
    if fc is not None:
        axes[0].axvspan(t0, t1, color=fc, alpha=fa, zorder=0)

axes[0].plot(t_sig, signal, lw=0.4, color='k', alpha=0.9, zorder=2)
axes[0].set_ylim(ymin, ymax)
axes[0].set_ylabel('Amplitude', fontsize=20)
axes[0].set_title('Audio Waveform (background: manual Ground Truth)', fontsize=16, loc='left')
axes[0].tick_params(axis='both', labelsize=19)
axes[0].margins(x=0)

# ── Panel 2: DNN ─────────────────────────────────────────────────────────
_draw_timeline(axes[1], dnn_segs)
axes[1].set_ylim(0, 1)
axes[1].set_yticks([])
axes[1].set_ylabel('DNN Model\nVAD', fontsize=20, fontweight='bold',
                   rotation=0, labelpad=75, va='center')
axes[1].set_xlabel('Time (s)', fontsize=20)
axes[1].tick_params(axis='x', labelsize=19)
axes[1].margins(x=0)

# ── Shared legend ─────────────────────────────────────────────────────────
legend_patches = [Patch(fc='cyan', ec='none', alpha=1.0, label='Single speaker'),
                  Patch(fc='orange', ec='none', alpha=1.0, label='Overlap')]
fig.legend(handles=legend_patches, loc='upper right', ncol=2,
           framealpha=1.0, bbox_to_anchor=(0.98, 0.96),
           prop={'weight': 'bold', 'size': 20})

fig.subplots_adjust(left=0.08, right=0.98, top=0.93, bottom=0.05)
plt.show()

## 6. Sliding-Window Filtering — 1 s Window

Two sliding-window checks are applied (both use a 1-second centred window, **different thresholds per class**):

1. **Overlap → Single speaker:** For each frame labeled as **overlap (2)**, if fewer than **60 %** of the frames in the window are overlap, the label is changed to **single speaker (1)**.
2. **Single speaker → Silence:** For each frame labeled as **single speaker (1)**, if fewer than **40 %** of the frames in the window are single speaker, the label is changed to **silence (0)**.

Both checks read from the **original** (unmodified) labels, so they do not influence each other.

In [ ]:
# ── Overlap sliding-window filter ─────────────────────────────────────────
OVERLAP_THRESHOLD_SEC6 = 0.70   # overlap (2) — need ≥ 60 % in window to keep
SINGLE_THRESHOLD_SEC6  = 0.30   # single  (1) — need ≥ 40 % in window to keep


def sliding_window_filter(labels, hop_sec, window_sec=1.0,
                          overlap_threshold=OVERLAP_THRESHOLD_SEC6,
                          single_threshold=SINGLE_THRESHOLD_SEC6):
    """
    For every frame labelled as overlap (2), check a centred window of
    *window_sec* seconds.  If the fraction of overlap frames in the window
    is below *overlap_threshold*, re-label that frame as single speaker (1).

    For every frame labelled as single speaker (1), check a centred window
    of *window_sec* seconds.  If the fraction of single-speaker frames in
    the window is below *single_threshold*, re-label that frame as silence (0).

    Both checks read from the original (unmodified) labels.
    """
    filtered = labels.copy()
    half_win = int((window_sec / 2) / hop_sec)   # frames on each side
    n = len(labels)
    for i in range(n):
        if labels[i] == 2:
            lo = max(0, i - half_win)
            hi = min(n, i + half_win + 1)
            window = labels[lo:hi]
            if np.sum(window == 2) / len(window) < overlap_threshold:
                filtered[i] = 1
        elif labels[i] == 1:
            lo = max(0, i - half_win)
            hi = min(n, i + half_win + 1)
            window = labels[lo:hi]
            if np.sum(window == 1) / len(window) < single_threshold:
                filtered[i] = 0
    return filtered


trad_filtered = sliding_window_filter(trad_labels, hop_s,     window_sec=1.0,
                                      overlap_threshold=OVERLAP_THRESHOLD_SEC6,
                                      single_threshold=SINGLE_THRESHOLD_SEC6)
dnn_filtered  = sliding_window_filter(dnn_labels,  dnn_hop_s, window_sec=1.0,
                                      overlap_threshold=OVERLAP_THRESHOLD_SEC6,
                                      single_threshold=SINGLE_THRESHOLD_SEC6)

# ── Statistics ────────────────────────────────────────────────────────────
for name, orig, filt, hs in [('Traditional', trad_labels, trad_filtered, hop_s),
                              ('DNN',         dnn_labels,  dnn_filtered, dnn_hop_s)]:
    n_orig_ovl = np.sum(orig == 2)
    n_filt_ovl = np.sum(filt == 2)
    removed = n_orig_ovl - n_filt_ovl
    print(f'{name:12s}  overlap: {n_orig_ovl} → {n_filt_ovl}  '
          f'(removed {removed} frames = {removed * hs:.2f} s)')

# ── Build filtered segments ──────────────────────────────────────────────
trad_filt_segs = _label_segments(trad_filtered, hop_s)
dnn_filt_segs  = _label_segments(dnn_filtered,  dnn_hop_s)

# ── Figure (same layout as Section 5) ────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(22, 13), sharex=True,
                         gridspec_kw={'height_ratios': [1, 1],
                                      'hspace': 0.15})
fig.suptitle(f'Multivoice VAD Predictions (filtered) — {INPUT_WAV}',
             fontsize=28, fontweight='bold', y=0.985)

# ── Panel 1: Waveform with GT ────────────────────────────────────────────
t_sig = np.arange(len(signal)) / sr
gt_segs = _gt_sample_segments(gt_vad, sr)
ymin = -1.05 * np.max(np.abs(signal))
ymax =  1.05 * np.max(np.abs(signal))
for t0, t1, lab in gt_segs:
    fc, fa = GT_FILL[lab]
    if fc is not None:
        axes[0].axvspan(t0, t1, color=fc, alpha=fa, zorder=0)
axes[0].plot(t_sig, signal, lw=0.4, color='k', alpha=0.9, zorder=2)
axes[0].set_ylim(ymin, ymax)
axes[0].set_ylabel('Amplitude', fontsize=20)
axes[0].set_title('Audio Waveform (background: manual Ground Truth)', fontsize=16, loc='left')
axes[0].tick_params(axis='both', labelsize=19)
axes[0].margins(x=0)

# ── Panel 2: DNN (filtered) ──────────────────────────────────────────────
_draw_timeline(axes[1], dnn_filt_segs)
axes[1].set_ylim(0, 1)
axes[1].set_yticks([])
axes[1].set_ylabel('DNN Model\nVAD\n(1s window,\nO≥70% S≥30%)', fontsize=20, fontweight='bold',
                   rotation=0, labelpad=90, va='center')
axes[1].set_xlabel('Time (s)', fontsize=20)
axes[1].tick_params(axis='x', labelsize=19)
axes[1].margins(x=0)

# ── Shared legend ─────────────────────────────────────────────────────────
legend_patches = [Patch(fc='cyan', ec='none', alpha=1.0, label='Single speaker'),
                  Patch(fc='orange', ec='none', alpha=1.0, label='Overlap')]
fig.legend(handles=legend_patches, loc='upper right', ncol=2,
           framealpha=1.0, bbox_to_anchor=(0.98, 0.96),
           prop={'weight': 'bold', 'size': 20})

fig.subplots_adjust(left=0.08, right=0.98, top=0.93, bottom=0.05)
plt.show()

## 7. Hold / Sustain Smoothing — 500 ms

Applied on top of the **Section 6 filtered labels** (sliding-window filter).

Each active label is **held** (sustained) for an additional 500 ms to the right:

1. **Overlap hold (pass 1):** every frame labelled **overlap (2)** is extended 500 ms to the right, overwriting silence and single-speaker frames.
2. **Single-speaker hold (pass 2):** every frame labelled **single speaker (1)** (after pass 1) is extended 500 ms to the right, but **only into silence (0)**. The hold **stops** when an overlap (2) frame is encountered — overlap is never overwritten by single-speaker.

Both passes scan left → right; overlap has strict priority over single-speaker.

In [ ]:
# ── Hold / Sustain filter ────────────────────────────────────────────────
HOLD_MS = 750   # hold duration for both overlap and single-speaker


def hold_filter(labels, hop_sec, hold_ms=HOLD_MS):
    """
    Sustain (hold) filter — extends overlap and single-speaker labels
    to the right by *hold_ms* milliseconds.

    Two passes (left → right):
      1. **Overlap hold:** every frame labelled 2 is held for *hold_ms*
         to the right, overwriting silence and single-speaker.
      2. **Single-speaker hold:** every frame labelled 1 (after pass 1)
         is held for *hold_ms* to the right, but only into silence (0).
         The hold stops when an overlap (2) frame is encountered.

    Overlap has priority — single-speaker never overwrites overlap.
    """
    hold_frames = max(1, int(hold_ms / 1000 / hop_sec))
    filtered = labels.copy()
    n = len(filtered)

    # Pass 1: extend overlap to the right
    counter = 0
    for i in range(n):
        if filtered[i] == 2:
            counter = hold_frames
        elif counter > 0:
            filtered[i] = 2
            counter -= 1

    # Pass 2: extend single-speaker to the right (stop at overlap)
    counter = 0
    for i in range(n):
        if filtered[i] == 2:
            counter = 0          # overlap encountered → stop single hold
        elif filtered[i] == 1:
            counter = hold_frames
        elif counter > 0:        # filtered[i] == 0 (silence)
            filtered[i] = 1
            counter -= 1

    return filtered


# ── Apply to filtered labels from Section 6 ──────────────────────────────
trad_held = hold_filter(trad_filtered, hop_s)
dnn_held  = hold_filter(dnn_filtered,  dnn_hop_s)

# ── Statistics ────────────────────────────────────────────────────────────
print(f'Hold duration: {HOLD_MS} ms')
print(f'Input: Section 6 filtered labels (sliding-window filter)\n')
for name, filt, held, hs in [
        ('Traditional', trad_filtered, trad_held, hop_s),
        ('DNN',         dnn_filtered,  dnn_held,  dnn_hop_s)]:
    for lab, lname in [(0, 'Silence'), (1, 'Single'), (2, 'Overlap')]:
        c_before = np.sum(filt == lab)
        c_after  = np.sum(held == lab)
        diff = c_after - c_before
        print(f'{name:12s}  {lname:8s}: {c_before:6d} → {c_after:6d}  '
              f'({diff:+6d} frames = {diff * hs:+.2f} s)')
    print()

# ── Build segments ───────────────────────────────────────────────────────
trad_held_segs = _label_segments(trad_held, hop_s)
dnn_held_segs  = _label_segments(dnn_held,  dnn_hop_s)

# ── Figure ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(22, 13), sharex=True,
                         gridspec_kw={'height_ratios': [1, 1],
                                      'hspace': 0.15})
fig.suptitle(
    f'Multivoice VAD Predictions (hold {HOLD_MS} ms) — {INPUT_WAV}',
    fontsize=28, fontweight='bold', y=0.985)

# ── Panel 1: Waveform with GT ────────────────────────────────────────────
t_sig = np.arange(len(signal)) / sr
gt_segs = _gt_sample_segments(gt_vad, sr)
ymin = -1.05 * np.max(np.abs(signal))
ymax =  1.05 * np.max(np.abs(signal))
for t0, t1, lab in gt_segs:
    fc, fa = GT_FILL[lab]
    if fc is not None:
        axes[0].axvspan(t0, t1, color=fc, alpha=fa, zorder=0)
axes[0].plot(t_sig, signal, lw=0.4, color='k', alpha=0.9, zorder=2)
axes[0].set_ylim(ymin, ymax)
axes[0].set_ylabel('Amplitude', fontsize=20)
axes[0].set_title('Audio Waveform (background: manual Ground Truth)', fontsize=16, loc='left')
axes[0].tick_params(axis='both', labelsize=19)
axes[0].margins(x=0)

# ── Panel 2: DNN (held) ─────────────────────────────────────────────────
_draw_timeline(axes[1], dnn_held_segs)
axes[1].set_ylim(0, 1)
axes[1].set_yticks([])
axes[1].set_ylabel(f'DNN Model\nVAD\n(1s window,\nO≥70% S≥30%)\n[hold {HOLD_MS} ms]',
                   fontsize=20, fontweight='bold',
                   rotation=0, labelpad=90, va='center')
axes[1].set_xlabel('Time (s)', fontsize=20)
axes[1].tick_params(axis='x', labelsize=19)
axes[1].margins(x=0)

# ── Shared legend ─────────────────────────────────────────────────────────
legend_patches = [Patch(fc='cyan', ec='none', alpha=1.0, label='Single speaker'),
                  Patch(fc='orange', ec='none', alpha=1.0, label='Overlap')]
fig.legend(handles=legend_patches, loc='upper right', ncol=2,
           framealpha=1.0, bbox_to_anchor=(0.98, 0.96),
           prop={'weight': 'bold', 'size': 20})

fig.subplots_adjust(left=0.08, right=0.98, top=0.93, bottom=0.05)
plt.show()